# Prioritization of PF4 Variants Across Cardiovascular GWAS Datasets

This notebook identifies prioritized PF4-region variants by integrating cardiovascular GWAS association results with PF4 variant annotations from NCBI dbSNP.

In [12]:
from pathlib import Path

import pandas as pd

In [13]:
associations_path = Path("../data/final/cardiovascular_associations.csv")
variants_path = Path("../data/final/pf4_variants.csv")

out_dir = Path("../data/final")
out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / "prioritized_variants.csv"

In [14]:
associations_df = pd.read_csv(associations_path)
variants_df = pd.read_csv(variants_path)

print("Associations shape:", associations_df.shape)
print("Variants shape:", variants_df.shape)

Associations shape: (2614, 20)
Variants shape: (1976, 18)


In [15]:
variant_columns = [
    "rsID",
    "Priority",
    "MAX_freq",
    "Location_relative_to_PF4",
    "Variant_type",
    "Functional_consequence"
]

variants_subset = variants_df[variant_columns].copy()

variants_subset.head()

,rsID,Priority,MAX_freq,Location_relative_to_PF4,Variant_type,Functional_consequence
0,rs2233648,Primary,0.005184,Inside,snv,"coding_sequence_variant,synonymous_variant"
1,rs367951530,Below threshold,0.000058,Inside,snv,"coding_sequence_variant,missense_variant"
2,rs764474600,Below threshold,0.000013,Inside,snv,"coding_sequence_variant,missense_variant"
3,rs765704070,Below threshold,0.000000,Inside,snv,"coding_sequence_variant,missense_variant"
4,rs982761496,Below threshold,0.000000,Inside,snv,"coding_sequence_variant,missense_variant"


In [16]:
prioritized_rsids = variants_subset[
    variants_subset["Priority"].isin(["Primary", "Secondary"])
]["rsID"].unique()

print("Number of prioritized rsIDs:", len(prioritized_rsids))

Number of prioritized rsIDs: 176


In [17]:
prioritized_df = associations_df[
    associations_df["rsID"].isin(prioritized_rsids)
].copy()

print("Rows after prioritized rsID filtering:", prioritized_df.shape[0])

prioritized_df.head()

Rows after prioritized rsID filtering: 154


,source_dataset,phenotype,trait_name,association_id,variant_id,rsID,chromosome,position,effect_allele,other_allele,effect_size,standard_error,odds_ratio,ci_lower,ci_upper,p_value,allele_frequency,mapped_genes,nearest_genes,is_selected_trait
1,GWAS Catalog,NaN,growth-regulated alpha protein measurement,104535614.0,NaN,rs11574450,NaN,NaN,A,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.950000,PF4,NaN,False
7,GWAS Catalog,NaN,platelet factor 4 level,188346583.0,NaN,rs11574450,NaN,NaN,C,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e-28,0.047187,PF4,NaN,False
8,GWAS Catalog,NaN,blood protein amount,94162087.0,NaN,rs3756074,NaN,NaN,C,NaN,NaN,NaN,NaN,NaN,NaN,8.000000e-26,0.027140,PF4; PPBP,NaN,False
9,GWAS Catalog,NaN,annexin A6 measurement,104509584.0,NaN,rs3756074,NaN,NaN,C,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e-24,0.050000,PF4; PPBP,NaN,False
10,GWAS Catalog,NaN,blood protein amount,94106139.0,NaN,rs3756074,NaN,NaN,C,NaN,NaN,NaN,NaN,NaN,NaN,9.000000e-23,0.027140,PF4; PPBP,NaN,False


In [18]:
prioritized_df = prioritized_df.merge(
    variants_subset,
    on="rsID",
    how="left"
)

print("Merged prioritized shape:", prioritized_df.shape)

prioritized_df.head()

Merged prioritized shape: (154, 25)


,source_dataset,phenotype,trait_name,association_id,variant_id,rsID,chromosome,position,effect_allele,other_allele,...,p_value,allele_frequency,mapped_genes,nearest_genes,is_selected_trait,Priority,MAX_freq,Location_relative_to_PF4,Variant_type,Functional_consequence
0,GWAS Catalog,NaN,growth-regulated alpha protein measurement,104535614.0,NaN,rs11574450,NaN,NaN,A,NaN,...,0.000000e+00,0.950000,PF4,NaN,False,Primary,0.115542,Inside,snv,intron_variant
1,GWAS Catalog,NaN,platelet factor 4 level,188346583.0,NaN,rs11574450,NaN,NaN,C,NaN,...,1.000000e-28,0.047187,PF4,NaN,False,Primary,0.115542,Inside,snv,intron_variant
2,GWAS Catalog,NaN,blood protein amount,94162087.0,NaN,rs3756074,NaN,NaN,C,NaN,...,8.000000e-26,0.027140,PF4; PPBP,NaN,False,Primary,0.117814,Near,snv,"2KB_upstream_variant,upstream_transcript_variant"
3,GWAS Catalog,NaN,annexin A6 measurement,104509584.0,NaN,rs3756074,NaN,NaN,C,NaN,...,1.000000e-24,0.050000,PF4; PPBP,NaN,False,Primary,0.117814,Near,snv,"2KB_upstream_variant,upstream_transcript_variant"
4,GWAS Catalog,NaN,blood protein amount,94106139.0,NaN,rs3756074,NaN,NaN,C,NaN,...,9.000000e-23,0.027140,PF4; PPBP,NaN,False,Primary,0.117814,Near,snv,"2KB_upstream_variant,upstream_transcript_variant"


In [19]:
numeric_columns = [
    "p_value",
    "effect_size",
    "odds_ratio",
    "allele_frequency"
]

for col in numeric_columns:
    if col in prioritized_df.columns:
        prioritized_df[col] = pd.to_numeric(
            prioritized_df[col],
            errors="coerce"
        )

In [20]:
prioritized_df = prioritized_df.sort_values(
    ["rsID", "p_value"],
    ascending=[True, True],
    na_position="last"
)

prioritized_df.head()

,source_dataset,phenotype,trait_name,association_id,variant_id,rsID,chromosome,position,effect_allele,other_allele,...,p_value,allele_frequency,mapped_genes,nearest_genes,is_selected_trait,Priority,MAX_freq,Location_relative_to_PF4,Variant_type,Functional_consequence
126,FinnGen,Coronary artery disease,Coronary artery disease,NaN,NaN,rs112860339,4.0,73982651.0,T,C,...,4.493650e-01,0.003965,NaN,PF4,NaN,Primary,0.007006,Near,snv,"2KB_upstream_variant,upstream_transcript_variant"
132,FinnGen,Myocardial infarction,Myocardial infarction,NaN,NaN,rs112860339,4.0,73982651.0,T,C,...,5.695420e-01,0.004005,NaN,PF4,NaN,Primary,0.007006,Near,snv,"2KB_upstream_variant,upstream_transcript_variant"
140,FinnGen,Heart failure,Heart failure,NaN,NaN,rs112860339,4.0,73982651.0,T,C,...,7.532060e-01,0.003965,NaN,PF4,NaN,Primary,0.007006,Near,snv,"2KB_upstream_variant,upstream_transcript_variant"
0,GWAS Catalog,NaN,growth-regulated alpha protein measurement,104535614.0,NaN,rs11574450,NaN,NaN,A,NaN,...,0.000000e+00,0.950000,PF4,NaN,False,Primary,0.115542,Inside,snv,intron_variant
1,GWAS Catalog,NaN,platelet factor 4 level,188346583.0,NaN,rs11574450,NaN,NaN,C,NaN,...,1.000000e-28,0.047187,PF4,NaN,False,Primary,0.115542,Inside,snv,intron_variant


In [21]:
prioritized_df.to_csv(out_csv, index=False)

print("Saved prioritized variants table:")
print(out_csv)

Saved prioritized variants table:
../data/final/prioritized_variants.csv


In [22]:
run_summary = {
    "output_file": str(out_csv),
    "total_rows": int(prioritized_df.shape[0]),
    "unique_rsIDs": int(prioritized_df["rsID"].nunique()),
    "priority_counts": prioritized_df["Priority"].value_counts(dropna=False).to_dict(),
    "rows_by_dataset": prioritized_df["source_dataset"].value_counts(dropna=False).to_dict()
}

run_summary

{'output_file': '../data/final/prioritized_variants.csv',
 'total_rows': 154,
 'unique_rsIDs': 15,
 'priority_counts': {'Primary': 151, 'Secondary': 3},
 'rows_by_dataset': {'GWAS Catalog': 88,
  'FinnGen': 45,
  'CARDIoGRAMplusC4D': 16,
  'HERMES': 5}}